<a href="https://colab.research.google.com/github/melissa-04/spink1-gbm-hypoxia/blob/main/03_transcriptome_gsea.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 Transcriptome-wide correlation and GSEA

Computes the Spearman correlation of every expressed gene with SPINK1 in the discovery cohort and runs pre-ranked GSEA against the 50 MSigDB hallmark gene sets. GSEA p and FDR values of zero (no permutation reached the observed value) are reported as <0.001, the resolution of 1,000 permutations. Produces Table S4 and Table S5.

In [1]:
%%capture
!pip install -q gseapy

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import gseapy as gp
from scipy import stats
from statsmodels.stats.multitest import multipletests

REPO = '/content/drive/MyDrive/SPINK1/repo'
RESULTS = os.path.join(REPO, 'results')
L = pd.read_parquet(os.path.join(REPO, 'processed', 'tcga_expression.parquet'))
print(L.shape, '| gseapy', gp.__version__)

Mounted at /content/drive
(17754, 144) | gseapy 1.3.2


In [3]:
R = L.T.rank()
Rc = R - R.mean()
ss = np.sqrt((Rc ** 2).sum())
rho = Rc.mul(Rc['SPINK1'], axis=0).sum() / (ss * ss['SPINK1'])
rho = rho.drop('SPINK1').replace([np.inf, -np.inf], np.nan).dropna()

n = L.shape[1]
t = rho * np.sqrt((n - 2) / (1 - rho ** 2))
s4 = pd.DataFrame({'gene': rho.index, 'rho': rho.values,
                   'p': 2 * stats.t.sf(np.abs(t), df=n - 2)})
s4['FDR_q'] = multipletests(s4['p'], method='fdr_bh')[1]
s4 = s4.sort_values('rho', ascending=False).reset_index(drop=True)
s4.to_csv(os.path.join(RESULTS, 'Table_S4_SPINK1_genomewide_correlation.csv'), index=False)

sig = s4[s4['FDR_q'] < 0.05]
print('Genes tested:', len(s4), '| FDR < 0.05:', len(sig),
      f'({(sig.rho > 0).sum()} positive, {(sig.rho < 0).sum()} negative)')
FMT = {'rho': '{:.3f}'.format, 'p': '{:.2e}'.format, 'FDR_q': '{:.2e}'.format}
print(s4.head(10).to_string(index=False, formatters=FMT))
print(s4.tail(10).iloc[::-1].to_string(index=False, formatters=FMT))

Genes tested: 17753 | FDR < 0.05: 4158 (2581 positive, 1577 negative)
    gene   rho        p    FDR_q
     IL8 0.664 1.12e-19 1.99e-15
   TREM1 0.617 1.92e-16 1.71e-12
 C5orf46 0.597 2.97e-15 1.76e-11
   CCL20 0.594 4.52e-15 2.01e-11
     PI3 0.585 1.36e-14 4.36e-11
C15orf48 0.584 1.61e-14 4.36e-11
   NDRG1 0.583 1.72e-14 4.36e-11
  INSIG2 0.582 1.98e-14 4.39e-11
  S100A9 0.578 3.52e-14 6.94e-11
  ENTHD1 0.568 1.16e-13 2.04e-10
    gene    rho        p    FDR_q
   ATAD2 -0.484 7.86e-10 2.15e-07
    MCM4 -0.468 3.35e-09 7.08e-07
    LIG1 -0.462 5.46e-09 1.01e-06
 GATAD2B -0.459 7.49e-09 1.28e-06
   DNMT1 -0.449 1.69e-08 2.40e-06
   XRCC3 -0.448 1.82e-08 2.52e-06
    MSH2 -0.440 3.55e-08 4.26e-06
   BRIP1 -0.439 3.78e-08 4.49e-06
TIMELESS -0.434 5.33e-08 5.77e-06
    ILF3 -0.431 7.08e-08 7.29e-06


In [4]:
pre = gp.prerank(rnk=s4[['gene', 'rho']], gene_sets='MSigDB_Hallmark_2020',
                 permutation_num=1000, seed=42, min_size=10, max_size=500,
                 outdir=None, verbose=False)

s5 = pre.res2d.copy()
for c in ['NES', 'NOM p-val', 'FDR q-val']:
    s5[c] = pd.to_numeric(s5[c])
s5 = s5.sort_values('NES', ascending=False).reset_index(drop=True)
s5 = s5[['Term', 'ES', 'NES', 'NOM p-val', 'FDR q-val', 'Tag %', 'Lead_genes']]
n_sig = int((s5['FDR q-val'] < 0.05).sum())

bound = lambda x: '<0.001' if x == 0 else x
out = s5.assign(**{c: s5[c].map(bound) for c in ['NOM p-val', 'FDR q-val']})
out.to_csv(os.path.join(RESULTS, 'Table_S5_GSEA_hallmark.csv'), index=False)

print('Gene sets:', len(s5), '| FDR < 0.05:', n_sig)
view = out[['Term', 'NES', 'NOM p-val', 'FDR q-val']].copy()
view['NES'] = view['NES'].round(3)
for c in ['NOM p-val', 'FDR q-val']:
    view[c] = view[c].map(lambda x: x if isinstance(x, str) else f'{x:.3g}')
print(view.to_string(index=False))

2026-10-30 20:24:33,597 [WARNING] Duplicated values found in preranked stats: 1.68% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


Gene sets: 50 | FDR < 0.05: 40
                             Term    NES NOM p-val FDR q-val
    TNF-alpha Signaling via NF-kB  3.062    <0.001    <0.001
            Inflammatory Response  2.956    <0.001    <0.001
                          Hypoxia  2.801    <0.001    <0.001
         IL-6/JAK/STAT3 Signaling  2.763    <0.001    <0.001
              Allograft Rejection  2.724    <0.001    <0.001
                      Coagulation  2.535    <0.001    <0.001
Epithelial Mesenchymal Transition  2.531    <0.001    <0.001
                       Complement  2.519    <0.001    <0.001
                KRAS Signaling Up  2.334    <0.001    <0.001
                        Apoptosis  2.214    <0.001    <0.001
  Reactive Oxygen Species Pathway  2.189    <0.001    <0.001
             IL-2/STAT5 Signaling  2.185    <0.001    <0.001
                      p53 Pathway  2.067    <0.001    <0.001
                       Glycolysis  2.057    <0.001    <0.001
                     Angiogenesis  2.054    <0.001    